In [1]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & artiq_run " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, img_amp, freq_raman):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu
        from kexp.util.artiq.async_print import aprint

        class hf_monitored_rabi(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.DISPERSIVE)

                self.p.frequency_raman_transition = {freq_raman}

                self.p.t_continuous_rabi = 400.e-6

                # self.xvar('t_raman_pulse',[0, 8.7e-06 / 2, 8.7e-06])

                # self.xvar('t_raman_pulse',np.linspace(0.,8.7e-6,7))
                self.p.t_raman_pulse = 8.8699e-6
                
                # self.xvar('amp_imaging',np.linspace(0.2,1.,10))
                self.p.amp_imaging = {img_amp}

                self.p.hf_imaging_detuning = -568.e6 # 182.

                self.p.hf_imaging_detuning_mid = -514.e6 # -1 --> 0
                
                self.p.dimension_slm_mask = 20.e-6

                self.p.phase_slm_mask = 0.186 * np.pi

                self.p.t_tweezer_hold = 20.e-3

                self.p.t_tof = 20.e-6
                
                self.p.t_mot_load = 1.0
                
                self.p.N_repeats = 20

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):
                
                self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning_mid)
                # self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning)
                self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask,dimension=self.p.dimension_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(squeeze=True)

                self.raman.init(fraction_power = self.p.fraction_power_raman,
                                frequency_transition = self.p.frequency_raman_transition)

                self.ttl.raman_shutter.on()
                delay(10.e-3)
                self.ttl.line_trigger.wait_for_line_trigger()
                delay(4.7e-3)

                # self.raman.pulse(t=self.p.t_raman_pulse)
                
                self.ttl.pd_scope_trig3.pulse(1.e-6)
                self.imaging.on()
                delay(1.e-6)
                self.raman.pulse(t=self.p.t_continuous_rabi)
                # delay(self.p.t_continuous_rabi)
                self.imaging.off()

                self.ttl.raman_shutter.off()
                
                self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning)
                self.imaging.set_power(.2,reset_pid=True)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

                self.core.wait_until_mu(now_mu())
                self.scope.read_sweep(0)
                self.core.break_realtime()
                delay(30.e-3)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                self.mot_observe()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                # aprint(self.scope._data)
                self.end(expt_filepath)
                """)
        return script

In [2]:
eBuilder = ExptBuilder()

In [ ]:
img_amp = np.linspace(.2,2.5, 100)
# np.random.shuffle(ts)
freq_raman = np.array([1.19499214e+08, 1.19503446e+08, 1.19507677e+08, 1.19511908e+08,
       1.19516139e+08, 1.19520371e+08, 1.19524602e+08, 1.19528833e+08,
       1.19533064e+08, 1.19537296e+08, 1.19541527e+08, 1.19545758e+08,
       1.19549990e+08, 1.19554221e+08, 1.19558452e+08, 1.19562683e+08,
       1.19566915e+08, 1.19571146e+08, 1.19575377e+08, 1.19579608e+08,
       1.19583840e+08, 1.19588071e+08, 1.19592302e+08, 1.19596533e+08,
       1.19600765e+08, 1.19604996e+08, 1.19609227e+08, 1.19613459e+08,
       1.19617690e+08, 1.19621921e+08, 1.19626152e+08, 1.19630384e+08,
       1.19634615e+08, 1.19638846e+08, 1.19643077e+08, 1.19647309e+08,
       1.19651540e+08, 1.19655771e+08, 1.19660003e+08, 1.19664234e+08,
       1.19668465e+08, 1.19672696e+08, 1.19676928e+08, 1.19681159e+08,
       1.19685390e+08, 1.19689621e+08, 1.19693853e+08, 1.19698084e+08,
       1.19702315e+08, 1.19706547e+08, 1.19710778e+08, 1.19715009e+08,
       1.19719240e+08, 1.19723472e+08, 1.19727703e+08, 1.19731934e+08,
       1.19736165e+08, 1.19740397e+08, 1.19744628e+08, 1.19748859e+08,
       1.19753090e+08, 1.19757322e+08, 1.19761553e+08, 1.19765784e+08,
       1.19770016e+08, 1.19774247e+08, 1.19778478e+08, 1.19782709e+08,
       1.19786941e+08, 1.19791172e+08, 1.19795403e+08, 1.19799634e+08,
       1.19803866e+08, 1.19808097e+08, 1.19812328e+08, 1.19816560e+08,
       1.19820791e+08, 1.19825022e+08, 1.19829253e+08, 1.19833485e+08,
       1.19837716e+08, 1.19841947e+08, 1.19846178e+08, 1.19850410e+08,
       1.19854641e+08, 1.19858872e+08, 1.19863103e+08, 1.19867335e+08,
       1.19871566e+08, 1.19875797e+08, 1.19880029e+08, 1.19884260e+08,
       1.19888491e+08, 1.19892722e+08, 1.19896954e+08, 1.19901185e+08,
       1.19905416e+08, 1.19909647e+08, 1.19913879e+08, 1.19918110e+08])
for f in range(len(img_amp)):
    print(img_amp[f],freq_raman[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(img_amp=img_amp[f],freq_raman = freq_raman[f]))
    eBuilder.run_expt()

0.2 119499214.0
0 Run id: 65766
 20 values of dummy. 20 total shots. 60 total images expected.
B:\_K\PotassiumData\2026-05-02\0065766_2026-05-02_10-31-06_hf_monitored_rabi.hdf5
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.186 pi, x-center = 993, y-center = 820

 Run ID: 65766

Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.186 pi, x-center = 993, y-center = 820


Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.186 pi, x-center = 993, y-center = 820


Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle

In [13]:

relay = ethernet_relay

In [14]:
relay.source_off()

AttributeError: module 'waxx.control.ethernet_relay' has no attribute 'source_off'